<a href="https://colab.research.google.com/github/prabhnoor14/ds2002-fa26/blob/main/notebooks/04-json-apis/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [11]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.ConnectionError, requests.Timeout) as e:
            print(f'attempt {attempt + 1}, network error, retrying')
            r = None
        if r is not None:
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                # 400 / 404 etc. mean our request is wrong, so retrying won't help
                raise ValueError(f'bad request ({r.status_code}): {r.text[:200]}')
            print(f'attempt {attempt + 1}: got {r.status_code}, will retry')
        if attempt < tries - 1:
            time.sleep(2 ** attempt)   # 1s, 2s, 4s ... grows each time
    raise RuntimeError(f'get_weather failed after {tries} attempts')

Explanation: there are 3 tries, and i set timeout to 10 seconds. datafrmae returned on 200, wait doubles on the retires for 429/5xx, value error on other statuses. network errors also get retried, 400/404 raise right away because retrying won’t help, and if all 3 tries fail it raises a RuntimeError

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [12]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
DATES = ['2026-09-05', '2026-10-03']

try:
    raw = get_weather('2026-09-05', '2026-10-03')
    wx = raw[raw['time'].isin(DATES)].reset_index(drop=True)
except RuntimeError as e:
    print('API failed:', e)
    wx = None

wx

,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
1,2026-10-03,22.0,14.7


Explaination: i made one request for a the range from sep 5 to oct 3 keeping only the game day rows. i caught runtime error so outage would lead to fallback. a ValueError (our bug) isn’t caught, so it still crashes instead of quietly using fake data.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [13]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

if wx is None:
    used_fallback = True
    wx = FALLBACK.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


Explaination: switch to sample data when the api fails and have a flag for it so we still have data but can tell its sample. every row gets a source column so anyone can tell if it’s real or sample

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [14]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
wx.dtypes

,0
date,datetime64[ns]
temp_max,float64
precip_mm,float64
source,object


Explaination: I renamed the API's columns to our own short names, turned date from text into a  datetime so it will match other tables in a join, and fixed the column order

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [15]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


explaination: confirmed the table has the right columns, exactly two rows, a real datetime column, and no missing or negative rainfall, so any problem shows up now instead of later inside a join.

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [16]:
# TODO
wet = wx.loc[wx['precip_mm'].idxmax()]
dry = wx.loc[wx['precip_mm'].idxmin()]
diff = wet['precip_mm'] - dry['precip_mm']
note = ' (NOTE: this uses sample data, not real weather)' if used_fallback else ''

if diff == 0:
    print(f"Both home weekends got the same rain: {wet['precip_mm']:.1f} mm.{note}")
else:
    print(f"{wet['date']:%B %d, %Y} was the wetter home weekend, with "
          f"{wet['precip_mm']:.1f} mm of rain compared to {dry['precip_mm']:.1f} mm "
          f"on {dry['date']:%B %d, %Y}, which is {diff:.1f} mm more.{note}")

October 03, 2026 was the wetter home weekend, with 14.7 mm of rain compared to 2.5 mm on September 05, 2026, which is 12.2 mm more.


explaination: i found the rows with the most and least rain, subtract them, and print dates and amounts, plus a warning if the numbers came from the fallback. October 3 was the wetter day

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [17]:
# TODO
pulled_at = pd.Timestamp.now()

print('Source:      ', 'Open-Meteo Historical Weather API' if not used_fallback else 'fallback sample (NOT real data)')
print('Endpoint:    ', BASE)
print('Coordinates:  lat 38.03, lon -78.51 ')
print('Date range:   2026-09-05 to 2026-10-03')
print('Pulled at:   ', pulled_at.strftime('%Y-%m-%d %H:%M:%S'))
print('Real or fallback:', 'fallback sample' if used_fallback else 'real (open-meteo)')

Source:       Open-Meteo Historical Weather API
Endpoint:     https://archive-api.open-meteo.com/v1/archive
Coordinates:  lat 38.03, lon -78.51 
Date range:   2026-09-05 to 2026-10-03
Pulled at:    2026-10-12 04:01:47
Real or fallback: real (open-meteo)


explaination: This prints everything someone would need to re-pull the exact same data and shows whether the data was real or fallback

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame instead of an error, the code would keep running. The left join would make all rain values NaN, and filling them with 0 would make it look like there was no rain for all 200 days. Since the sales data still looks right, we might not notice the problem and think rain has no effect on revenue._your answer here_

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [18]:
# stretch
CITIES = {
    'Charlottesville': (38.03, -78.51),
    'Orlando': (28.54, -81.38),
}

frames = []
for city, (lat, lon) in CITIES.items():
    try:
        raw = get_weather('2026-09-05', '2026-10-03', lat=lat, lon=lon)
    except RuntimeError as e:
        print(f'{city}: API failed ({e}), skipping. No fake data added.')
        continue
    df = raw[raw['time'].isin(DATES)].copy()
    df['city'] = city
    df['source'] = 'open-meteo'
    frames.append(df)

if frames:
    both = pd.concat(frames, ignore_index=True)
    both = both.rename(columns={'time': 'date',
                                'temperature_2m_max': 'temp_max',
                                'precipitation_sum': 'precip_mm'})
    both['date'] = pd.to_datetime(both['date'])
    both = both[['city', 'date', 'temp_max', 'precip_mm', 'source']]
    print(both)
else:
    print('No city data could be pulled.')

              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo
2          Orlando 2026-09-05      32.1        3.9  open-meteo
3          Orlando 2026-10-03      30.4       14.5  open-meteo


explaination: We call the same function for each city, add a city column, and combine the results into one table, skipping any city that fails. Both cities had more rain on October 3 than September 5. Orlando was also hotter on both days, with a bigger temperature difference in October.